# 07 · Testing DAGs

> **Module 2 · Reliability & Operations · Lecture 2.5**

In this notebook you will:

- Split testing into three layers: **pure logic**, **DAG structure**, **integration**
- Write structure tests against a tiny Airflow shim so they run without Airflow
- Use `pytest.mark.parametrize` to test every branch of a decision function
- Learn what **not** to test — and why testing Airflow itself is a mistake
- Assemble a CI checklist you can put in a pipeline tomorrow

## 1. Why DAGs need tests at all

A DAG file is code. It runs on a schedule, in production, often unattended.
Yet most DAG code is tested not at all — which is strange, because a DAG is
*easier* to test than most application code: it is a declarative graph, and
the graph has structure you can assert on.

There are three layers, and they have very different costs:

| layer | what it tests | needs Airflow? | typical runtime |
|---|---|---|---|
| **1. Pure logic** | functions in `core/` | no | milliseconds |
| **2. DAG structure** | task ids, dependencies, trigger rules, timeouts | yes, or a shim | seconds |
| **3. Integration** | does it run against real GCP? | yes + credentials | minutes, expensive |

The ordering matters more than the tooling. Get layer 1 right and layers 2
and 3 become cheap. Skip layer 1 and you are testing BigQuery to find out
whether your rounding is correct.

> **Put the logic in functions, not in tasks.** A task should be a thin
> wrapper: build the context, call a function, return the result. The
> function is testable; the wrapper is not.

This is why every module in `project/core/` imports nothing from Airflow.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

print("COURSE_ROOT =", COURSE_ROOT)

import importlib.util
print("pytest importable:", importlib.util.find_spec("pytest") is not None)
print("airflow importable:", importlib.util.find_spec("airflow") is not None)
print()

# The modules under test. None of them import Airflow.
import pipeline_logic
import templating
import fanout
import sla_logic
import sla_tracker
import sensor_conditions
import idempotent_load
import dedup_load

for module in (pipeline_logic, templating, fanout, sla_logic, sla_tracker,
               sensor_conditions, idempotent_load, dedup_load):
    print(f"  {module.__name__:<20} {module.__file__}")

## 2. Layer 1 — testing pure logic

This is the highest-value, lowest-cost testing you will do. The functions
take data and return data. There is no scheduler, no worker, no clock and no
network.

Two techniques carry most of the weight.

### Parametrise instead of repeating

```python
@pytest.mark.parametrize(
    "row_count,threshold,expected",
    [(0, 100, False), (100, 100, False), (101, 100, True)],
)
def test_should_run_full_load(row_count, threshold, expected):
    assert should_run_full_load(row_count, threshold) == expected
```

One test, three cases, and — critically — the **boundary** is included. The
`>` versus `>=` bug lives exactly at `row_count == threshold`, so that case
must be in the table. A test that only tries `0` and `1000` would not catch
it.

### Assert on the reason, not just the outcome

When a row is rejected, the *reason code* is the contract. Asserting
`accepted == 4705` tells you the count is wrong; asserting that a specific
row is rejected with reason `bad_timestamp` tells you **why**, and lets the
data be fixed.

Let's build both.

In [ ]:
"""A miniature test framework, so we can see what pytest does.

In a real project you would use pytest directly. This module exists so the
notebook can demonstrate the *shape* of a parametrised test - and so the
exercise has a home for the helper functions.
"""
from __future__ import annotations


class AssertionFailed(Exception):
    """Raised when a test assertion fails."""


def check(condition, message=""):
    """Assert a condition.

    Args:
        condition: The boolean condition.
        message: An optional failure message.

    Raises:
        AssertionFailed: When the condition is false.
    """
    if not condition:
        raise AssertionFailed(message or "assertion failed")


def check_equal(actual, expected, message=""):
    """Assert equality.

    Args:
        actual: The value under test.
        expected: The expected value.
        message: An optional failure message.

    Raises:
        AssertionFailed: When the values differ.
    """
    if actual != expected:
        raise AssertionFailed(
            message or f"expected {expected!r}, got {actual!r}"
        )


def run_cases(name, fn, cases):
    """Run a test function over a list of argument tuples.

    Args:
        name: The test's name.
        fn: The function under test.
        cases: A list of positional-argument tuples.

    Returns:
        The number of cases that passed.
    """
    passed = 0
    for case in cases:
        try:
            fn(*case)
            passed += 1
            print(f"    PASS  {name}{case}")
        except (AssertionFailed, AssertionError) as exc:
            print(f"    FAIL  {name}{case}: {exc}")
    return passed

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

from pipeline_logic import should_run_full_load

# The helper defined in the previous cell is still in the namespace.
def test_should_run_full_load(row_count, threshold, expected):
    """The full-load decision must be a strict greater-than."""
    actual = should_run_full_load(row_count, threshold)
    assert actual == expected, f"expected {expected}, got {actual}"


CASES = [
    (0, 100, False),
    (99, 100, False),
    (100, 100, False),   # boundary: NOT greater than
    (101, 100, True),
    (5000, 100, True),
]

passed = 0
for case in CASES:
    try:
        test_should_run_full_load(*case)
        passed += 1
        print(f"    PASS  test_should_run_full_load{case}")
    except AssertionError as exc:
        print(f"    FAIL  test_should_run_full_load{case}: {exc}")

print()
print(f"{passed}/{len(CASES)} cases passed")
print()
print("The boundary case (100, 100) is the one that catches the")
print("off-by-one between '>' and '>='.")

## 3. The Airflow shim

Layer 2 tests want to assert things like "the `load_complete` task uses a
trigger rule that survives a skipped branch". That requires a DAG object.

Requiring Airflow just to assert on a trigger rule is a heavy price: your
CI needs Airflow, its providers, and a compatible Python. The compromise is
a **shim** — a tiny stand-in with the same shape — that lets structure
tests run on a bare Python install.

The shim below provides just enough: `DAG` with `tasks`, `Task` with
`task_id`, `upstream_task_ids`, `trigger_rule`, `retries` and `timeout`, and
`dag.get_task(id)`.

Then a structure test is just Python:

```python
def test_join_survives_skipped_branch():
    join = dag.get_task("load_complete")
    assert join.trigger_rule in ("none_failed", "all_done")
```

The same test in a real Airflow environment imports the real DAG from a
`DagBag` and asserts the same thing. The *assertion* is identical; only the
source of the object changes. That is the whole trick.

In [ ]:
"""A minimal Airflow stand-in, enough to assert on DAG structure.

Mirrors the shape of airflow.models.DAG / airflow.models.BaseOperator for
the attributes a structure test cares about.
"""
from __future__ import annotations


class BaseOperator:
    """A stand-in for an Airflow operator."""

    def __init__(self, task_id, *, upstream_task_ids=None, trigger_rule="all_success",
                 retries=0, timeout=None, **kwargs):
        self.task_id = task_id
        self.upstream_task_ids = set(upstream_task_ids or ())
        self.trigger_rule = trigger_rule
        self.retries = retries
        self.timeout = timeout
        self.kwargs = kwargs

    def __repr__(self):
        return f"<Task {self.task_id}>"


class DAG:
    """A stand-in for an Airflow DAG."""

    def __init__(self, dag_id, tasks, *, schedule=None, start_date=None,
                 catchup=False, max_active_runs=1, default_args=None, **kwargs):
        self.dag_id = dag_id
        self.schedule = schedule
        self.start_date = start_date
        self.catchup = catchup
        self.max_active_runs = max_active_runs
        self.default_args = default_args or {}
        self._tasks = {task.task_id: task for task in tasks}

    def get_task(self, task_id):
        """Return a task by id.

        Args:
            task_id: The task's identifier.

        Returns:
            The task.

        Raises:
            KeyError: When no such task exists.
        """
        return self._tasks[task_id]

    @property
    def tasks(self):
        """Return all tasks."""
        return list(self._tasks.values())

    @property
    def task_ids(self):
        """Return all task ids."""
        return list(self._tasks)

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

# The branching DAG's shape, reconstructed from the deployable definition.
tasks = [
    BaseOperator("wait_for_upstream_data", trigger_rule="all_success", retries=3),
    BaseOperator("branch_on_volume", upstream_task_ids=["wait_for_upstream_data"]),
    BaseOperator("full_load", upstream_task_ids=["branch_on_volume"]),
    BaseOperator("incremental_load", upstream_task_ids=["branch_on_volume"]),
    BaseOperator("load_complete", upstream_task_ids=["full_load", "incremental_load"],
                 trigger_rule="none_failed"),
]

dag = DAG("branching_and_xcom", tasks, catchup=False, max_active_runs=1,
          default_args={"owner": "data-platform", "retries": 2})

# --- structure assertions ------------------------------------------------
assert dag.get_task("load_complete").trigger_rule in (
    "none_failed", "none_failed_min_one_success", "all_done"
), "the join point must survive a skipped branch"

assert dag.get_task("full_load").upstream_task_ids == {"branch_on_volume"}
assert dag.get_task("incremental_load").upstream_task_ids == {"branch_on_volume"}
assert dag.get_task("branch_on_volume").upstream_task_ids == {"wait_for_upstream_data"}
assert dag.get_task("wait_for_upstream_data").upstream_task_ids == set()

assert dag.catchup is False
assert dag.max_active_runs == 1
assert dag.default_args["retries"] >= 1

print("Structure assertions on the branching DAG:")
for task in dag.tasks:
    upstream = ", ".join(sorted(task.upstream_task_ids)) or "-"
    print(f"  {task.task_id:<24} <- {upstream:<24} trigger={task.trigger_rule}")
print()
print("PASS: join point uses 'none_failed' and survives a skipped branch.")

## 4. A reusable structure validator

Writing assertions inline works for one DAG. For a course, a project, or a
team, you want a **validator**: a function that checks a whole DAG against a
set of rules and returns the problems.

The rules that catch the most real bugs:

1. **No orphan tasks** — every task except the roots has upstream tasks.
2. **No cycles** — the graph is a DAG.
3. **Every join has a resilient trigger rule** — a join whose upstream can be
   skipped must not use `all_success`.
4. **Every task has a timeout** — a hung task must not hold a worker forever.
5. **Retries are bounded** — no task retries 99 times.
6. **The schedule is not None** unless the DAG is deliberately manual.

Rule 3 is the subtle one and it is worth stating explicitly: a join task
with `trigger_rule="all_success"` will **not** run when one of its upstreams
was skipped by a branch. That is the classic "my downstream task silently
never runs" bug.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

RESILIENT_TRIGGER_RULES = {"none_failed", "none_failed_min_one_success", "all_done"}


def validate_dag(dag, *, require_schedule=False, max_retries=5):
    """Check a DAG against a set of structural rules.

    Args:
        dag: The DAG to validate.
        require_schedule: Whether the DAG must have a schedule.
        max_retries: The highest acceptable retry count.

    Returns:
        A list of problem strings; empty means the DAG is valid.
    """
    problems = []

    # Rule 3: joins must survive a skipped upstream.
    for task in dag.tasks:
        if len(task.upstream_task_ids) > 1 and task.trigger_rule == "all_success":
            problems.append(
                f"task '{task.task_id}' joins {sorted(task.upstream_task_ids)} "
                f"with trigger_rule='all_success' - it will be skipped if "
                f"one branch is skipped"
            )

    # Rule 4/5: bounded timeouts and retries.
    for task in dag.tasks:
        if task.timeout is None:
            problems.append(f"task '{task.task_id}' has no timeout")
        if task.retries > max_retries:
            problems.append(
                f"task '{task.task_id}' retries {task.retries} times "
                f"(max {max_retries})"
            )

    # Rule 6: schedule.
    if require_schedule and dag.schedule is None:
        problems.append("DAG has no schedule")

    return problems


# A healthy DAG.
good = DAG(
    "good_dag",
    [
        BaseOperator("extract", timeout=600, retries=2),
        BaseOperator("transform", upstream_task_ids=["extract"], timeout=600, retries=2),
        BaseOperator("load", upstream_task_ids=["transform"], timeout=1800, retries=1),
    ],
    schedule="@daily",
)

# A broken DAG: a fragile join, no timeout, absurd retries, no schedule.
bad = DAG(
    "bad_dag",
    [
        BaseOperator("branch", timeout=300, retries=0),
        BaseOperator("path_a", upstream_task_ids=["branch"], timeout=300, retries=0),
        BaseOperator("path_b", upstream_task_ids=["branch"], timeout=300, retries=0),
        BaseOperator("join", upstream_task_ids=["path_a", "path_b"],
                     trigger_rule="all_success", retries=99),
        BaseOperator("no_timeout", upstream_task_ids=["join"]),
    ],
)

print("good_dag problems:", validate_dag(good, require_schedule=True) or "none")
print()
print("bad_dag problems:")
for problem in validate_dag(bad, require_schedule=True):
    print(f"  - {problem}")

## 5. Asserting on rejection reasons

Back to layer 1, because it is worth doing properly. When
`transform_orders` rejects a row, the reason code is the contract with
whoever owns the upstream system.

```python
def test_malformed_rows_report_specific_reasons():
    accepted, rejected = transform_orders(malformed_lines)
    reasons = {row["reason"] for row in rejected}
    assert "missing_order_id" in reasons
    assert "bad_timestamp" in reasons
```

Asserting only `len(rejected) == 5` would still pass if all five rows were
rejected for the *wrong* reason — a bug that sends every alert to the wrong
team.

The same reasoning applies to the happy path: assert the **exact** expected
summary, not just the row count.

```python
assert summarise_by_region(accepted) == EXPECTED_SUMMARY
```

An equality assertion against a full expected value catches every field at
once. A test that checks `summary["AMER"]["revenue_eur"] > 1000` catches
almost nothing.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = Path.cwd()
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent
if not (COURSE_ROOT / "project").exists():
    COURSE_ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(COURSE_ROOT / "project" / "core"))
sys.path.insert(0, str(COURSE_ROOT / "project"))

from pipeline_logic import transform_orders, summarise_by_region

SAMPLE_DIR = COURSE_ROOT / "sample_data"
orders = (SAMPLE_DIR / "orders.csv").read_text(encoding="utf-8").splitlines()
malformed = (SAMPLE_DIR / "malformed_orders.csv").read_text(encoding="utf-8").splitlines()

# --- the happy path: exact equality, not "close enough" --------------------
accepted, rejected = transform_orders(orders)
summary = summarise_by_region(accepted)

EXPECTED_SUMMARY = {
    "AMER": {"order_count": 318, "revenue_eur": 113592.19},
    "APAC": {"order_count": 285, "revenue_eur": 108194.92},
    "EMEA": {"order_count": 322, "revenue_eur": 118423.22},
    "LATAM": {"order_count": 320, "revenue_eur": 110646.18},
}

assert summary == EXPECTED_SUMMARY, f"summary changed: {summary}"
assert len(accepted) == 1245
assert len(rejected) == 0
print("PASS: summary matches the expected value exactly.")
print()

# --- the unhappy path: assert on the reason codes ------------------------
accepted_bad, rejected_bad = transform_orders(malformed)
reasons = [row["reason"] for row in rejected_bad]

print(f"malformed rows rejected: {len(rejected_bad)}")
for row in rejected_bad:
    print(f"  reason={row['reason']:<20} detail={row['detail']}")

assert len(rejected_bad) == len(malformed) - 1
assert all(reason for reason in reasons), "every rejection needs a reason"
assert len(set(reasons)) == len(reasons), "each malformed row has a distinct reason"

print()
print("PASS: every rejection carries a specific reason code.")

## 6. What *not* to test

This is the part experienced teams learn the hard way.

**Do not test Airflow.** Your test should assert that *you* configured the
DAG correctly, not that `BaseOperator.__init__` stores what you passed it.
The moment your test asserts `task.retries == 3`, you are testing Airflow.
Test the *policy* instead: "the ETL DAG sets retries >= 1", "the sensor uses
`mode='reschedule'`".

**Do not test that a function runs.** `assert transform_orders([]) is not
None` proves only that Python executed a line.

**Do not mock the thing you are trying to verify.** Mocking BigQuery to test
a BigQuery load job proves the mock works.

**Do not test the golden path only.** A suite that never exercises a
rejection, a boundary or an empty input will pass while the pipeline is
broken in exactly the places that matter.

**Do not make integration tests a gate on every commit.** They cost
minutes and money. Run them nightly or before release; keep the commit gate
fast.

## 7. The CI checklist

```yaml
# .github/workflows/dags.yml
name: dags
on: [push, pull_request]

jobs:
  unit:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install pytest
      - run: pytest project/tests -q          # pure logic + shim structure tests
      - run: python resources/verify_notebooks.py --no-exec   # notebook syntax
  integration:
    if: github.ref == 'refs/heads/main'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - run: pip install pytest apache-airflow
      - run: pytest project/tests -q -m integration
```

## Exercises

1. **Boundary table.** Write a parametrised test for `should_run_full_load` covering `(0, 100)`, `(100, 100)`, `(101, 100)` and one larger case, and state in one sentence why the `100 == 100` case must be in the table.
2. **Harder — a structure rule.** Extend `validate_dag` with a rule that every task must have at least one downstream task or be an explicit "sink", and write a DAG that violates it. Then explain why `upstream_task_ids == set()` is not by itself an error.
3. **Harder — the CI gate.** Write a test that asserts the ETL DAG's load task uses `WRITE_TRUNCATE` rather than `WRITE_APPEND`. Explain why this assertion belongs in the DAG-structure layer and not in the pure-logic layer.

<details>
<summary><b>Solution (click to reveal)</b></summary>

**1.**

```python
@pytest.mark.parametrize("row_count,threshold,expected", [
    (0, 100, False),
    (100, 100, False),
    (101, 100, True),
    (1000, 100, True),
])
def test_should_run_full_load(row_count, threshold, expected):
    assert should_run_full_load(row_count, threshold) is expected
```

The `100 == 100` case must be included because the implementation uses a strict `>`; an off-by-one that changed it to `>=` would flip exactly that case and nowhere else. A table containing only `0` and `1000` would let that bug through.

**2.** Track, for each task, whether it appears in any other task's `upstream_task_ids`. Any task with no downstream is a candidate sink; flag it unless it is in an explicit `sinks` set. A violating DAG has `extract -> transform` with nothing downstream of `transform`.

`upstream_task_ids == set()` is not itself an error: a root task (the first task of a DAG) legitimately has no upstream. Roots are exactly what you want. The rule is about *sinks*, and even sinks are legitimate — a DAG that ends in a terminal task is the normal case — so the rule exists to make the choice explicit rather than to forbid it.

**3.**

```python
def test_etl_load_is_truncate(dagbag):
    dag = dagbag.dags["etl_orders_to_bigquery"]
    load = dag.get_task("load_to_bigquery")
    assert load.configuration["query"]["writeDisposition"] == "WRITE_TRUNCATE"
```

It belongs in the structure layer because `WRITE_TRUNCATE` is a property of the *task's configuration* — it lives in the DAG definition, not in any function under `core/`. The pure-logic layer cannot see it, because the pure-logic layer never touches the DAG. Testing it there would require inventing a fake, and the test would then assert only that the fake behaves as the fake was written — which is why "don't mock the thing you are verifying" is a rule and not a preference.

</details>